In [ ]:
spark

SparkSession - hive 
 
 
 SparkContext 

 Spark UI 

 
 Version 
 v3.3.2 
 Master 
 local[8] 
 AppName 
 Databricks Shell

In [ ]:
path = r"/FileStore/batch16_employee.csv"
emp_df = spark.read.csv(path, header = True, inferSchema = True)
emp_df.display()

employee_name,department,state,salary,age,bonus
James,Sales,NY,90000,34,10000
Michael,Sales,NY,86000,56,20000
Robert,Sales,CA,81000,30,23000
Maria,Finance,CA,90000,24,23000
Raman,Finance,CA,99000,40,24000
Scott,Finance,NY,83000,36,19000
Jen,Finance,NY,79000,53,15000
Jeff,Marketing,CA,80000,25,18000
Kumar,Marketing,NY,91000,50,21000


In [ ]:
# count number of employees from each department ..groupby..
df=emp_df.groupby("department").count()
df.display()

department,count
Sales,3
Finance,4
Marketing,2


In [ ]:
df= emp_df.groupBy("department").sum()
df.display()

department,sum(salary),sum(age),sum(bonus)
Sales,257000,120,53000
Finance,351000,153,81000
Marketing,171000,75,39000


In [ ]:
# average salary of employee departments
df = emp_df.groupBy("department").avg("salary")
df.display()

department,avg(salary)
Sales,85666.66666666667
Finance,87750.0
Marketing,85500.0


In [ ]:
# statewise gruping
df = emp_df.groupBy("state").count()
df.display()

state,count
CA,4
NY,5


In [ ]:
# min and max age of employees statewise...

df = emp_df.groupBy("state").min("age")
df1 = emp_df.groupBy("state").max("age")
df.display()
df1.display()

state,min(age)
CA,24
NY,34


state,max(age)
CA,40
NY,56


In [ ]:
print("Number of partitions in df:",emp_df.rdd.getNumPartitions())

Number of partitions in df: 1


In [ ]:
# increase number of partitions 
new_df = emp_df.repartition(6)
print("Number of partitions in df:",new_df.rdd.getNumPartitions())

Number of partitions in df: 6


In [ ]:
df1 = new_df.groupBy("state").min("age")
df1.display()

state,min(age)
NY,34
CA,24


In [ ]:
from pyspark.sql.functions import *
df2 = new_df.orderBy(col("age").desc())
df2.display()

employee_name,department,state,salary,age,bonus
Michael,Sales,NY,86000,56,20000
Jen,Finance,NY,79000,53,15000
Kumar,Marketing,NY,91000,50,21000
Raman,Finance,CA,99000,40,24000
Scott,Finance,NY,83000,36,19000
James,Sales,NY,90000,34,10000
Robert,Sales,CA,81000,30,23000
Jeff,Marketing,CA,80000,25,18000
Maria,Finance,CA,90000,24,23000


In [ ]:
# Add column with updated age..

df3 = new_df.withColumn("Updated Age",new_df.age+2)
df3.display()

employee_name,department,state,salary,age,bonus,Updated Age
Kumar,Marketing,NY,91000,50,21000,52
Jen,Finance,NY,79000,53,15000,55
Raman,Finance,CA,99000,40,24000,42
James,Sales,NY,90000,34,10000,36
Scott,Finance,NY,83000,36,19000,38
Robert,Sales,CA,81000,30,23000,32
Michael,Sales,NY,86000,56,20000,58
Maria,Finance,CA,90000,24,23000,26
Jeff,Marketing,CA,80000,25,18000,27


In [ ]:
# Fillna..

path =r"/FileStore/small_zipcode.csv"
df4 = spark.read.csv(path, header = True, inferSchema = True)
df4.display()

id,zipcode,type,city,state,population
1,704,STANDARD,null,PR,30100
2,704,null,PASEO COSTA DEL SUR,PR,null
3,709,null,BDA SAN LUIS,PR,3700
4,76166,UNIQUE,CINGULAR WIRELESS,TX,84000
5,76177,STANDARD,null,TX,null


In [ ]:
df5 = df4.na.fill(" ")
df4.display()

id,zipcode,type,city,state,population
1,704,STANDARD,null,PR,30100
2,704,null,PASEO COSTA DEL SUR,PR,null
3,709,null,BDA SAN LUIS,PR,3700
4,76166,UNIQUE,CINGULAR WIRELESS,TX,84000
5,76177,STANDARD,null,TX,null


In [ ]:
df = spark.read.csv(r"/FileStore/batch16.csv", header=True,inferSchema=True)
df.display()

Id,Name,Age,Department
1,a,18,aa
2,b,20,aa
3,c,18,aa
4,d,21,bb
5,e,22,bb
6,f,21,bb
7,g,21,cc
8,h,21,cc
9,i,24,cc


In [ ]:
from pyspark.sql.window import window
from pyspark.sql.functions import row_number
windowSpec = Window.partitionBy("Department").orderBy("Age")

df1 = df.withColumn("row_number",row_number().over(windowSpec))
df1.display()

---------------------------------------------------------------------------
ImportError                               Traceback (most recent call last)
File <command-3038140400030401>:1
----> 1 from pyspark.sql.window import window
      2 from pyspark.sql.functions import row_number
      3 windowSpec = window.partitionBy("Department").orderBy("Age")

ImportError: cannot import name 'window' from 'pyspark.sql.window' (/databricks/spark/python/pyspark/sql/window.py)

In [ ]:
from pyspark.sql.window import window
from pyspark.sql.functions import rank
windowSpec = Window.partitionBy("Department").orderBy("Age")

df1 = df.withColumn("Rank",rank().over(windowSpec))
df1.display()

---------------------------------------------------------------------------
ImportError                               Traceback (most recent call last)
File <command-3038140400030402>:1
----> 1 from pyspark.sql.window import window
      2 from pyspark.sql.functions import rank
      3 windowSpec = window.partitionBy("Department").orderBy("Age")

ImportError: cannot import name 'window' from 'pyspark.sql.window' (/databricks/spark/python/pyspark/sql/window.py)